In [39]:
!pip install tensorboard

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
!pip install langid

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [19]:
!pip install -U accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [20]:
import langid
text = "Shirika lilianzishwa tarehe 28 Machi 1884 na Karl Peters na Wajerumani wengine waliotaka Ujerumani kuingia kati ya mataifa yenye koloni. Shabaha ya shirika ilikuwa kuanzisha makoloni ya Kijerumani katika maeneo nje ya Ujerumani"
lang, score = langid.classify(text)
print(lang, score)

sw -314.5781469345093


In [21]:
from datasets import load_dataset

ds = load_dataset("google-research-datasets/tydiqa", "secondary_task")

In [22]:
ds.items()

dict_items([('train', Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 49881
})), ('validation', Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 5077
}))])

In [23]:
ds['train'][41000]

{'id': 'english-7062167395997684209-0',
 'title': 'Origin of Hangul',
 'context': 'The Korean alphabet is the native script of Korea, created in the mid fifteenth century by King Sejong,[1][2] as both a complement and an alternative to the logographic Sino-Korean hanja. Initially denounced by the educated class as eonmun (vernacular writing), it only became the primary Korean script following independence from Japan in the mid-20th century.[3]',
 'question': 'When was Hangul invented?',
 'answers': {'text': ['mid fifteenth century'], 'answer_start': [66]}}

In [24]:
ds['train']['id'][30069]

'arabic-7939514601024908244-0'

In [25]:
ids = ds['train']['id']

languages = [i.split('-')[0] for i in ids]
unique_languages = set(languages)
print(len(unique_languages))
print(unique_languages)

9
{'russian', 'finnish', 'arabic', 'telugu', 'english', 'korean', 'swahili', 'indonesian', 'bengali'}


In [26]:
from collections import Counter

lang_counts = Counter(languages)
print(lang_counts)

Counter({'arabic': 14805, 'finnish': 6855, 'russian': 6490, 'indonesian': 5702, 'telugu': 5563, 'english': 3696, 'swahili': 2755, 'bengali': 2390, 'korean': 1625})


In [27]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

### Example on a sample

In [28]:
sample = ds['train'][0]

question = sample['question']
context = sample['context']

inputs = tokenizer(
    question,
    context,
    truncation=True,
    padding='max_length',
    max_length=448
)
print(inputs.keys())

dict_keys(['input_ids', 'attention_mask'])


In [29]:
sample = ds['train'][41000]

answer_text = sample['answers']['text'][0]
answer_start = sample['answers']['answer_start'][0]

print(answer_text)
print(answer_start)
answer_end = answer_start + len(answer_text)
print(answer_end)

mid fifteenth century
66
87


In [30]:
inputs = tokenizer(
    sample['question'],
    sample['context'],
    truncation=True,
    padding='max_length',
    max_length=448,
    return_offsets_mapping=True
)

offsets = inputs['offset_mapping']
offsets

[(0, 0),
 (0, 4),
 (5, 8),
 (9, 13),
 (13, 15),
 (16, 22),
 (22, 24),
 (24, 25),
 (0, 0),
 (0, 0),
 (0, 3),
 (4, 10),
 (11, 12),
 (11, 19),
 (20, 22),
 (23, 26),
 (27, 29),
 (29, 33),
 (34, 40),
 (41, 43),
 (44, 49),
 (49, 50),
 (51, 58),
 (59, 61),
 (62, 65),
 (66, 69),
 (70, 72),
 (72, 75),
 (75, 77),
 (77, 79),
 (80, 87),
 (88, 90),
 (91, 95),
 (96, 98),
 (98, 102),
 (102, 103),
 (103, 109),
 (110, 112),
 (113, 117),
 (118, 119),
 (120, 130),
 (131, 134),
 (135, 137),
 (138, 149),
 (150, 152),
 (153, 156),
 (157, 161),
 (161, 168),
 (169, 173),
 (173, 174),
 (174, 179),
 (179, 180),
 (181, 184),
 (184, 186),
 (186, 187),
 (188, 190),
 (190, 193),
 (193, 197),
 (198, 201),
 (201, 206),
 (206, 207),
 (208, 210),
 (211, 214),
 (215, 221),
 (221, 223),
 (224, 229),
 (230, 232),
 (233, 234),
 (234, 236),
 (236, 239),
 (240, 241),
 (241, 244),
 (244, 246),
 (246, 251),
 (252, 259),
 (259, 261),
 (262, 264),
 (265, 269),
 (270, 276),
 (277, 280),
 (281, 288),
 (289, 295),
 (296, 302),
 (30

In [31]:
for i, (start, end) in enumerate(offsets):
    if start <= answer_start < end:
        start_token = i
    if start < answer_end <= end:
        end_token = i
print("start token:", start_token)
print("end_token:", end_token)

start token: 25
end_token: 30


In [32]:
decoded = tokenizer.decode(
    inputs['input_ids'][start_token:end_token+1]
)
print("Decoded:", decoded)
print("Actual:", answer_text)

Decoded: mid fifteenth century
Actual: mid fifteenth century


In [34]:
def preprocess(exp):

    inputs = tokenizer(
        exp['question'],
        exp['context'],
        truncation=True,
        padding='max_length',
        max_length=448,
        return_offsets_mapping=True
    )

    offsets = inputs['offset_mapping']

    answer_text = exp['answers']['text'][0]
    answer_start = exp['answers']['answer_start'][0]
    answer_end = answer_start + len(answer_text)

    start_token = None
    end_token = None

    for i, (start, end) in enumerate(offsets):
        if start == 0 and end == 0:
            continue

        if start <= answer_start < end:
            start_token = i
        if start < answer_end <= end:
            end_token = i

    if start_token is None or end_token is None:
        start_token, end_token = 0, 0

    inputs['start_positions'] = start_token
    inputs['end_positions'] = end_token

    inputs.pop('offset_mapping')

    return inputs

In [35]:
tokenized_dataset = ds.map(preprocess, batched =False)

In [40]:
tokenized_dataset = tokenized_dataset['train'].train_test_split(test_size=0.1)

### Model Training

In [36]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(
    'xlm-roberta-base'
)

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [37]:
from transformers import DefaultDataCollator
data_collator = DefaultDataCollator()

In [41]:
from transformers import TrainingArguments
from transformers import Trainer

training_args = TrainingArguments(
    output_dir='TheModel',
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_strategy='steps',
    logging_steps=100,
    logging_dir='./logs',
    save_steps=500,
    fp16=True,
    disable_tqdm=False,
    report_to='tensorboard'
)

In [42]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [ ]:
trainer.train()
trainer.save_model("TheModel")
tokeinzer.save_pretrained("TheModel")

Step,Training Loss
100,4.924900
200,4.146700
300,3.795000
400,3.523700
500,3.162000
600,2.781300
700,2.743800
800,2.394400
900,2.187400
1000,2.207600


In [6]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from transformers import DefaultDataCollator
from transformers import TrainingArguments, Trainer

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForQuestionAnswering.from_pretrained('/home/hatem/ml/Graduation projects NLP/2-Question answering bot /TheModel/checkpoint-7500')

In [8]:
from transformers import pipeline

qa_pipeline = pipeline(
    'question-answering',
    model=model,
    tokenizer=tokenizer,
    device=0
)

result = qa_pipeline(
    question="What is the capital of France?",
    context="France is a country in Europe. Its capital city is Paris, which is known for the Eiffel Tower."
)
print(result)

{'score': 0.8996502757072449, 'start': 51, 'end': 57, 'answer': 'Paris,'}


In [12]:
model.eval()

XLMRobertaForQuestionAnswering(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

XLMRobertaForQuestionAnswering(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias

In [18]:
def ask(question, context):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=448
    )

    # 🔥 move inputs to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits)

    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end+1],
        skip_special_tokens=True
    )

    return answer

In [21]:
context = """Mosasaurs were discovered in a limestone quarry at Maastricht on the Meuse.
They lived during the late Cretaceous period."""

question = "Where were the fossils discovered?"

print(ask("Where were the fossils discovered?", context))
print(ask("أين تم اكتشاف الحفريات؟", context))
print(ask("Где были найдены окаменелости?", context))

Maastricht on the Meuse
Maastricht on the Meuse
Maastricht on the Meuse


In [44]:
context = """خالد هاني يتمشى الي الحديقة  """
print(ask("who goes to the gardern ", context))

خالد هاني


In [45]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# =============================
# 🔥 LOAD MODEL
# =============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModelForQuestionAnswering.from_pretrained("./TheModel/checkpoint-7500")
model.to(device)
model.eval()

# =============================
# 🧠 QA FUNCTION
# =============================
def ask(question, context):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=448
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits)

    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end+1],
        skip_special_tokens=True
    )

    return answer

# =============================
# 🌍 EXAMPLE CONTEXTS
# =============================
examples = {
    "1": {
        "title": "🇬🇧 English - Science",
        "context": """Mosasaurs were discovered in a limestone quarry at Maastricht on the Meuse.
        They lived during the late Cretaceous period."""
    },
    "2": {
        "title": "🇸🇦 Arabic - History",
        "context": """القاهرة هي عاصمة مصر. يبلغ عدد سكانها أكثر من 20 مليون نسمة."""
    },
    "3": {
        "title": "🇷🇺 Russian - Science",
        "context": """Альберт Эйнштейн родился в Германии в 1879 году."""
    },
    "4": {
        "title": "🌍 Mixed - Multilingual",
        "context": """Paris is the capital of France. القاهرة هي عاصمة مصر.
        Berlin is in Germany."""
    }
}

# =============================
# 🎮 INTERFACE LOOP
# =============================
while True:
    print("\n==============================")
    print("🌍 Multilingual QA System")
    print("==============================")

    print("\nChoose a context:")
    for k, v in examples.items():
        print(f"{k}. {v['title']}")

    print("5. ✍️ Write your own context")
    print("0. ❌ Exit")

    choice = input("\nYour choice: ")

    if choice == "0":
        break

    if choice in examples:
        context = examples[choice]["context"]
        print("\n📄 Context:")
        print(context)
    elif choice == "5":
        context = input("\n✍️ Enter your context:\n")
    else:
        print("❌ Invalid choice")
        continue

    question = input("\n❓ Ask your question (any language):\n")

    answer = ask(question, context)

    print("\n🤖 Answer:")
    print(answer)


🌍 Multilingual QA System

Choose a context:
1. 🇬🇧 English - Science
2. 🇸🇦 Arabic - History
3. 🇷🇺 Russian - Science
4. 🌍 Mixed - Multilingual
5. ✍️ Write your own context
0. ❌ Exit



Your choice:  1



📄 Context:
Mosasaurs were discovered in a limestone quarry at Maastricht on the Meuse.
        They lived during the late Cretaceous period.



❓ Ask your question (any language):
 what period they lived in ?



🤖 Answer:
late Cretaceous

🌍 Multilingual QA System

Choose a context:
1. 🇬🇧 English - Science
2. 🇸🇦 Arabic - History
3. 🇷🇺 Russian - Science
4. 🌍 Mixed - Multilingual
5. ✍️ Write your own context
0. ❌ Exit


...


Your choice:  5

✍️ Enter your context:
 Israel is a terroist country and killed 200000 plastineiens

❓ Ask your question (any language):
 how many is killed



🤖 Answer:
200000

🌍 Multilingual QA System

Choose a context:
1. 🇬🇧 English - Science
2. 🇸🇦 Arabic - History
3. 🇷🇺 Russian - Science
4. 🌍 Mixed - Multilingual
5. ✍️ Write your own context
0. ❌ Exit



Your choice:  5

✍️ Enter your context:
 Islam is the truth. The kaaba locate in Makka and the prophet of islam is mohammad

❓ Ask your question (any language):
 who is the prophet of islam



🤖 Answer:
mohammad

🌍 Multilingual QA System

Choose a context:
1. 🇬🇧 English - Science
2. 🇸🇦 Arabic - History
3. 🇷🇺 Russian - Science
4. 🌍 Mixed - Multilingual
5. ✍️ Write your own context
0. ❌ Exit



Your choice:  0
